# Quantitative trader assignment

A trader’s day-to-day work involves extracting insights from data, suggesting improvements to existing trading strategies, and developing new ones.
This assignment covers both: Part 1 focuses on finding new strategies, and Part 2 focuses on improving live strategies. Each part includes its own dataset and questions to answer. You’re not limited to answering only our questions, there’s always more to discover, so feel free to share any additional insights you find.

Good luck, and keep in mind that your approach and thought process matter more than the final numeric results.

# Part 1: Finding arbitrage across markets

One goal of market making is to keep prices aligned. When prices start to diverge, arbitrage opportunities can arise: you can buy an asset where it’s cheaper and sell it where it’s more expensive, locking in a profit. An explanation can be found here: https://corporatefinanceinstitute.com/resources/foreign-exchange/triangular-arbitrage-opportunity/

In this exercise you will identify the arbitrage trades between two markets: BTC/USD and BTC/EUR? Also provided is the EUR/USD exchange rate. 

You can make the following assumptions:

- You can execute trades against the given prices.
- All initiated trades get fully filled.
- The trading fee is 0.01% for the BTC markets and 0.005% for EUR/USD.
- Trades happen instantaneously.
- Profits are expressed in EUR.

## Questions
- Given the dataset, how would you define an arbitrage opportunity?
- How many trade opportunities are there during this time window?
- Take the *best* trade opportunity (so there is at least one, lucky you) and explain the trade.
- What is the risk free profit of this opportunity (in EUR) to be made if given 1000 USD, 1000 EUR and 0.1 BTC?
- What should you keep in mind when using this strategy in practice?

## Data
As mentioned before, you are given market data of two markets and the exchange rate eur to dollar. The trades are stored in data/part_1. The data consist of a timestamp column recorded in UTC with a 1 second time interval and a single price column. 



In [133]:
"""Part 1."""

'Part 1.'

In [134]:
import numpy as np 
import pandas as pd 
import pickle 
import glob 
import matplotlib.pyplot as plt 

In [ ]:
with open(r"", "rb") as file:
    data = pickle.load(file)

df = pd.DataFrame(data)
df["time"] = pd.to_datetime(df["time"])

df.head()

,market,time,price,eur_usd_rate
0,BTC/EUR,2026-02-09 12:45:00+00:00,58181.60,1.187322
1,BTC/EUR,2026-02-09 12:45:04+00:00,58188.85,1.187322
2,BTC/EUR,2026-02-09 12:45:05+00:00,58191.45,1.187328
3,BTC/EUR,2026-02-09 12:45:06+00:00,58198.60,1.187332
4,BTC/EUR,2026-02-09 12:45:17+00:00,58183.40,1.187332


### Definition of Arbitrage Opportunity: 

$BTC/USD = BTC/EUR \cdot EUR/USD$

$Spread = BTC/USD - (BTC/EUR \cdot EUR/USD)$

if: 
- $Spread > 0 \rightarrow$ BTC/USD is overpriced $\rightarrow$ go short BTC/USD
- $Spread < 0 \rightarrow$ BTC/USD is underpriced $\rightarrow$ go long BTC/USD

### How many trading opportunities are there ? 

In [138]:
pivot = df.pivot_table(index='time', columns='market', values='price', aggfunc='last').reset_index()
fx = df.groupby('time', as_index=False)['eur_usd_rate'].last()
pivot = pivot.merge(fx, on='time', how='left').sort_values('time')

pivot[['BTC/EUR','BTC/USD','eur_usd_rate']] = pivot[['BTC/EUR','BTC/USD','eur_usd_rate']].ffill()
pivot = pivot.dropna(subset=['BTC/EUR','BTC/USD','eur_usd_rate'])

pivot['btc_usd_implied'] = pivot['BTC/EUR'] * pivot['eur_usd_rate']
pivot['spread'] = pivot['BTC/USD'] - pivot['btc_usd_implied']

pivot

,time,BTC/EUR,BTC/USD,eur_usd_rate,btc_usd_implied,spread
1,2026-02-09 12:45:02+00:00,58181.60,69076.000000,1.187325,69080.489173,-4.489173
2,2026-02-09 12:45:04+00:00,58188.85,69076.000000,1.187325,69089.097281,-13.097281
3,2026-02-09 12:45:05+00:00,58191.45,69076.000000,1.187328,69092.350299,-16.350299
4,2026-02-09 12:45:06+00:00,58198.60,69076.100000,1.187332,69101.088685,-24.988685
5,2026-02-09 12:45:08+00:00,58198.60,69090.073333,1.187322,69100.466213,-10.392880
...,...,...,...,...,...,...
356,2026-02-09 12:59:52+00:00,58166.70,69079.700000,1.187740,69086.936323,-7.236323
357,2026-02-09 12:59:53+00:00,58166.70,69079.700000,1.187740,69086.936323,-7.236323
358,2026-02-09 12:59:54+00:00,58166.70,69079.800000,1.187740,69086.888767,-7.088767
359,2026-02-09 12:59:58+00:00,58166.70,69079.800000,1.187740,69086.888767,-7.088767


In [139]:
pivot['rel_spread'] = pivot['spread'] / pivot['BTC/USD']
fee_btc = 0.0001
fee_eurusd = 0.00005

total_fee = fee_btc + fee_eurusd

pivot["net_rel_spread"] = abs(pivot["rel_spread"]) - total_fee

pivot['signal'] = 0

pivot.loc[pivot['rel_spread'] < -total_fee, 'signal'] = 1   # long BTC/USD vs synthetic
pivot.loc[pivot['rel_spread'] >  total_fee, 'signal'] = -1  # short BTC/USD vs synthetic

pivot['signal_shifted'] = pivot['signal'].shift(1).fillna(0)

pivot['new_trade'] = ( (pivot['signal'] != 0) & (pivot['signal_shifted'] == 0) )

num_trades = pivot['new_trade'].sum()
print(f"After accounting for transaction fees, there are {num_trades} trading opportunities. ")

After accounting for transaction fees, there are 28 trading opportunities. 


### Best Trade opportunity 

In [142]:
pivot.loc[pivot['rel_spread'].idxmin()]

time               2026-02-09 12:56:29+00:00
BTC/EUR                              58082.3
BTC/USD                         68944.333333
eur_usd_rate                        1.187729
btc_usd_implied                 68986.022046
spread                            -41.688713
rel_spread                         -0.000605
net_rel_spread                      0.000455
signal                                     1
signal_shifted                           1.0
new_trade                              False
Name: 277, dtype: object

The best Trading opportunity took place on February 9th 2026 at 12:56:29. On this day, the market values were the following: 

- $BTC/USD=68944.33 \$ $
- Implied $BTC/USD= 68986.022\$ $
- $Spread = -41.688 \$ $
- Relative Spread $= 0.0605\% $
- Net Relative Spread $=0.0455\%$
- $Z-Score = -2.8 \cdot \sigma$

Since the spread is negative, we buy **BTC/USD** at $68944.33$, and then sell synthetic **BTC/USD** at $68986.022$. 

### Risk-Free Profit

In [143]:
best_idx = pivot['net_rel_spread'].idxmax()
best_trade = pivot.loc[best_idx]

net_edge = best_trade['net_rel_spread']

profit_1000_usd = 1000 * net_edge

profit_1000_eur = 1000 * net_edge

btc_price = best_trade["BTC/USD"]

profit_01_btc = 0.1 * btc_price * net_edge 


print(f"Risk-free profit for $1000: ${profit_1000_usd}")
print(f"Risk-free profit for €1000: €{profit_1000_eur}")
print(f"Risk-free profit for 0.1 BTC: €{profit_01_btc}")

Risk-free profit for $1000: $0.5732353182048257
Risk-free profit for €1000: €0.5732353182048257
Risk-free profit for 0.1 BTC: €3.9578889236622636


### What to keep in mind when using this in practice ? 

There are a few components we need to keep in mind: 

- Liquidity risk: 
    - Order books are not unlimited so we should take that into account as well
- Latency risk: 
    - If the execution is too slow, then by the time the trade is executed, the arbitrage may have been resolved
    - This latency issue goes hand in hand with the data synchronization problem: we need to make sure the data updates itself systematically and as often as possible 
- Capital and Funding: 
    - This is not a self-financing strategy, therefore we need to consider the amount of capital that is available to us at the beginning of our trading period  
- Other Fees: 
    - Apart from trading fees, we also need to take into consideration taxes 
- Model risk:
    - How reliable is our model ? Has it been stress-tested ? Back-tested ? 

# Part 2: Gaining trading insights from trade data

You are given executed trade data and reference price data for two markets: BTC/USDT and ETH/USDT.
The datasets cover the same time period and allow you to evaluate trading performance at trade level. Your goal is to compute Profit and Loss (PnL) over different horizons, diagnose why one market is clearly underperforming, and propose concrete improvements to the trading strategy.

## Profit and Loss
A core metric tracked by traders is **Profit and Loss (PnL)**. In this assignment, you will compute PnL using two complementary approaches:
- **Theoretical PnL**: Based on executed trades and reference prices over a chosen time horizon
  (e.g. at trade time or 1 second later). This measures execution quality and short-term slippage. We calculate this PnL as follows:
$$
\mathrm{PnL}_t(\text{side}) =
\text{trade amount} \times s \times (\text{reference price}_t - \text{trade price})
- \text{fee amount} \times \text{trade price},
\quad
s =
\begin{cases}
+1 & \text{if side = buy} \\
-1 & \text{if side = sell}
\end{cases}
$$
- **Accounting PnL**: Based on realized cash flows from trades and the value of the final inventory at the end of the dataset. This represents the actual profit or loss of the strategy over the full trading period.

Both perspectives are important and may lead to different conclusions.

## Questions

Address the following questions in your analysis:

- Merge the trade data with the price data and compute the theoretical PnL for different time horizons (e.g. for 1, 5 and 30 seconds). What patterns do you observe?
- One of the two markets is clearly underperforming. Can you identify the main source of the PnL leak? What mistake is being made by us?
- Break down performance across different dimensions (e.g. side, trigger, indicator, maker/taker). What concrete changes would you recommend to improve the strategy?
- What do you think the `indicator` column represents?
- What is the accounting PnL over the full dataset time span? How does it compare to the theoretical PnL results?

## Data

The dataset consists of hourly pickle files containing:
- Executed trades with timestamps, prices, sizes, fees, sides, and metadata
- Reference price snapshots representing fair market prices

All timestamps are in UTC and all prices are denominated in USDT.

In [144]:
"""Part 2."""

'Part 2.'

### Merging the data 

In [145]:
price_btcusd = r"C:\Users\Startklar\Documents\Blosh Assignment\assignment-trader-role\data\Price Data - BTCUSD\*.pickle"
price_ethusd = r"C:\Users\Startklar\Documents\Blosh Assignment\assignment-trader-role\data\Price Data - ETHUSD\*.pickle"

trade_btcusd = r"C:\Users\Startklar\Documents\Blosh Assignment\assignment-trader-role\data\Trade Data - BTCUSD\*.pickle"
trade_ethusd = r"C:\Users\Startklar\Documents\Blosh Assignment\assignment-trader-role\data\Trade Data - ETHUSD\*.pickle"

def load_pickles(pattern):
    files = sorted(glob.glob(pattern))
    df_list = [pd.read_pickle(f) for f in files]
    return pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

btcusd_price = load_pickles(price_btcusd)
ethusd_price  = load_pickles(price_ethusd)
btcusd_trade  = load_pickles(trade_btcusd)
ethusd_trade  = load_pickles(trade_ethusd)

btcusd_price['time'] = pd.to_datetime(btcusd_price['time'])
btcusd_trade['time'] = pd.to_datetime(btcusd_trade['time'])
btcusd_price = btcusd_price.sort_values('time')
btcusd_trade = btcusd_trade.sort_values('time')

ethusd_price['time'] = pd.to_datetime(ethusd_price['time'])
ethusd_trade['time'] = pd.to_datetime(ethusd_trade['time'])
ethusd_price = ethusd_price.sort_values('time')
ethusd_trade = ethusd_trade.sort_values('time')

### Computing the Theoretical PnL 

In [146]:
def compute_pnl_for_horizon(trades, prices, horizon_seconds, price_tolerance='2s'):
    trades_temp = trades.copy()
    trades_temp['time'] = pd.to_datetime(trades_temp['time'])
    prices = prices.copy()
    prices['time'] = pd.to_datetime(prices['time'])

    trades_temp['s'] = trades_temp['taker_side'].map({'buy': 1, 'sell': -1})

    trades_temp['ref_time'] = trades_temp['time'] + pd.Timedelta(seconds=horizon_seconds)

    merged = pd.merge_asof( trades_temp.sort_values('ref_time'), prices[['time', 'mid_price']].sort_values('time'),
        left_on='ref_time', right_on='time', direction='forward', tolerance=pd.Timedelta(price_tolerance) )

    merged[f'pnl_{horizon_seconds}s'] = ( merged['trade_amount'] * merged['s'] * (merged['mid_price'] - merged['trade_price'])
        - merged['fee_amount'] * merged['trade_price'] )

    return merged[[f'pnl_{horizon_seconds}s']]


In [147]:
pnl_1s  = compute_pnl_for_horizon(btcusd_trade, btcusd_price, 1)
pnl_5s  = compute_pnl_for_horizon(btcusd_trade, btcusd_price, 5)
pnl_30s = compute_pnl_for_horizon(btcusd_trade, btcusd_price, 30)

btcusd_trade = pd.concat([btcusd_trade.reset_index(drop=True), pnl_1s, pnl_5s, pnl_30s], axis=1)

In [148]:
btcusd_trade[['pnl_1s','pnl_5s','pnl_30s']].describe()

,pnl_1s,pnl_5s,pnl_30s
count,975.000000,978.000000,974.000000
mean,-0.415166,-0.433823,-0.338132
std,3.230287,3.517812,3.023377
min,-80.626323,-87.795110,-68.752834
25%,-0.111815,-0.126291,-0.155356
50%,-0.039891,-0.035858,-0.026294
75%,-0.005820,-0.000634,0.031181
max,5.617709,8.738070,15.293857


In [149]:
pnl_1s_eth  = compute_pnl_for_horizon(ethusd_trade, ethusd_price, 1)
pnl_5s_eth  = compute_pnl_for_horizon(ethusd_trade, ethusd_price, 5)
pnl_30s_eth = compute_pnl_for_horizon(ethusd_trade, ethusd_price, 30)

ethusd_trade = pd.concat([ethusd_trade.reset_index(drop=True), pnl_1s_eth, pnl_5s_eth, pnl_30s_eth], axis=1)

ethusd_trade

,market,time,trade_price,trade_amount,original_order_amount,fee_amount,taker_side,own_side,trigger,counterparty_id,indicator,pnl_1s,pnl_5s,pnl_30s
0,ETH-USDT,2026-02-08 11:00:22.089694+00:00,2095.725016,0.045147,0.056434,0.000032,buy,sell,1.0,CP-009,NaN,-0.096930,-0.096820,-0.070174
1,ETH-USDT,2026-02-08 11:01:48.353257+00:00,2097.662724,0.111187,0.135922,0.000078,buy,sell,1.0,CP-008,NaN,-0.210565,-0.228644,-0.036332
2,ETH-USDT,2026-02-08 11:02:47.061344+00:00,2101.229068,0.055957,0.069768,0.000039,buy,sell,1.0,CP-009,0.000369,-0.130799,-0.096947,-0.148163
3,ETH-USDT,2026-02-08 11:03:08.600562+00:00,2101.226492,0.013985,0.018190,0.000010,buy,sell,2.0,CP-004,0.000516,-0.039014,-0.028026,-0.026797
4,ETH-USDT,2026-02-08 11:03:09.695745+00:00,2100.766782,0.005594,0.007478,0.000004,buy,sell,2.0,CP-004,0.000489,-0.012532,-0.008112,-0.008654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
718,ETH-USDT,2026-02-08 22:58:20.114636+00:00,2108.988675,0.597778,0.683056,0.000418,sell,buy,3.0,CP-011,NaN,-0.850298,-1.474710,-1.164219
719,ETH-USDT,2026-02-08 22:59:17.793061+00:00,2099.472818,NaN,NaN,NaN,buy,NaN,NaN,CP-004,NaN,NaN,NaN,NaN
720,ETH-USDT,2026-02-08 22:59:17.873061+00:00,2107.618663,0.237524,0.281326,0.000166,buy,sell,1.0,CP-004,0.002726,-0.316940,-0.262923,-0.227057
721,ETH-USDT,2026-02-08 22:59:17.873061+00:00,2108.078795,0.077009,0.090528,0.000054,buy,sell,2.0,CP-013,0.001488,-0.138216,-0.120702,-0.109074


In [150]:
ethusd_trade[['pnl_1s','pnl_5s','pnl_30s']].describe()

,pnl_1s,pnl_5s,pnl_30s
count,617.000000,619.000000,618.000000
mean,-0.594827,-0.518460,-0.605562
std,2.172832,2.226534,2.684349
min,-15.928295,-15.727873,-18.551629
25%,-0.681015,-0.678535,-0.774090
50%,-0.335175,-0.288081,-0.259954
75%,-0.083675,-0.061246,-0.036126
max,24.817057,24.580507,27.004801


Since all BTC/USD mean Theoretical PnL values are less negative than the ETH/USD ones, then we can conclude that the ETH/USD Market is underperforming.

The main mistake we are doing is that we are making the trades at the wrong moment. We are reacting to crossing the spread rather than predicting future price movements. This explains why the markets tends to move against us shortly afterwards.

Also, since we are liquidity takers, we always start in the 'red' as we are always paying the spread. 

### Breaking Down the Performance Across Different Dimensions 

#### **taker_side**

In [151]:
btcusd_trade.groupby('taker_side')[['pnl_1s','pnl_5s','pnl_30s']].mean()

,pnl_1s,pnl_5s,pnl_30s
taker_side,,,
buy,-0.442305,-0.446720,-0.366551
sell,-0.326757,-0.391877,-0.245677


In [152]:
ethusd_trade.groupby('taker_side')[['pnl_1s','pnl_5s','pnl_30s']].mean()

,pnl_1s,pnl_5s,pnl_30s
taker_side,,,
buy,-0.844582,-0.824104,-0.982816
sell,-0.344262,-0.217715,-0.230743


#### **trigger**

In [153]:
btcusd_trade.groupby('trigger')[['pnl_1s','pnl_5s','pnl_30s']].mean()

,pnl_1s,pnl_5s,pnl_30s
trigger,,,
1.0,-0.067982,-0.055270,-0.099480
2.0,-0.976656,-1.037859,-0.868677
3.0,-0.180220,-0.185424,-0.022788


In [154]:
ethusd_trade.groupby('trigger')[['pnl_1s','pnl_5s','pnl_30s']].mean()

,pnl_1s,pnl_5s,pnl_30s
trigger,,,
1.0,-0.354934,-0.353133,-0.327787
2.0,-0.204783,-0.119109,-0.138982
3.0,-1.187935,-1.051190,-1.314354


#### **indicator**

In [155]:
btcusd_trade.groupby('indicator')[['pnl_1s','pnl_5s','pnl_30s']].mean()

,pnl_1s,pnl_5s,pnl_30s
indicator,,,
0.000013,-0.049507,-0.035837,0.041806
0.000023,-0.015865,-0.026135,-0.007029
0.000032,-0.004003,-0.004003,0.002596
0.000042,-0.005534,-0.000211,0.010269
0.000043,0.005859,0.005859,0.009986
...,...,...,...
0.003058,0.476639,1.536322,0.264703
0.003102,0.869890,0.093647,1.175712
0.003198,-0.086368,-0.421207,-2.183705


In [156]:
ethusd_trade.groupby('indicator')[['pnl_1s','pnl_5s','pnl_30s']].mean()

,pnl_1s,pnl_5s,pnl_30s
indicator,,,
-9999.000000,-0.697087,-0.808064,-0.967021
0.000006,-7.770925,-7.730244,-11.547927
0.000016,-0.173585,-0.110555,-0.105348
0.000063,-0.170201,-0.087955,-0.101555
0.000080,-0.362381,-0.187268,-0.216226
...,...,...,...
0.004151,-0.968252,-0.553295,1.075459
0.004498,-1.228940,-1.836796,-1.261534
0.004644,-1.152269,-1.093198,-0.557655


#### Concrete changes to improve the strategy: 

- Since ETH/USD is performing worse than BTC/USD, maybe we could separate strategies for both and create a new one, specifically for ETH/USD
- Add a short-term confirmation filter: check the stock price momentum to confirm whether we should trigger a trade 
- Remove the worst-performing triggers 

### What do you think the indicator column represent ? 

In [157]:
btcusd_trade['indicator'].describe()

count                                      942
unique                                     940
top       ERROR: Failed to calculate indicator
freq                                         3
Name: indicator, dtype: object

In [158]:
ethusd_trade['indicator'].describe()

count      572.0
unique     570.0
top      -9999.0
freq         3.0
Name: indicator, dtype: float64

I suppose that this column represents some kind of market metric that has triggered the trade. 

For example, it could be the strength of the signal that triggered the trade it corresponds to. Given its range of values, it looks to be some kind of normalized signal, that could potentially represent a time-series momentum or a deviation from a reference price. 

### Accounting PnL: 

$Cash_{Final} + Inventory_{Final} * Price_{Final}$

In [163]:
btcusd_trade['s'] = btcusd_trade['taker_side'].map({'buy': 1, 'sell': -1})

btcusd_trade["signed_amount"] = btcusd_trade["trade_amount"] * btcusd_trade["s"]
btcusd_trade["inventory"] = btcusd_trade["signed_amount"].cumsum()
btcusd_trade["cash_flow"] = - btcusd_trade["trade_price"] * (btcusd_trade["signed_amount"] + btcusd_trade["fee_amount"])
btcusd_trade['cash'] = btcusd_trade['cash_flow'].cumsum()

final_price_btc = btcusd_price['mid_price'].iloc[-1]
final_inventory_btc = btcusd_trade['inventory'].iloc[-1]
final_cash_btc = btcusd_trade['cash'].iloc[-1]

accounting_pnl_btc = final_cash_btc + final_inventory_btc * final_price_btc

print(accounting_pnl_btc)

-2635.714208787889


In [166]:
ethusd_trade.head()

,market,time,trade_price,trade_amount,original_order_amount,fee_amount,taker_side,own_side,trigger,counterparty_id,indicator,pnl_1s,pnl_5s,pnl_30s,s,signed_amount,inventory,cash_flow,cash
0,ETH-USDT,2026-02-08 11:00:22.089694+00:00,2095.725016,0.045147,0.056434,0.000032,buy,sell,1.0,CP-009,NaN,-0.096930,-0.096820,-0.070174,1,0.045147,0.045147,-94.682432,-94.682432
1,ETH-USDT,2026-02-08 11:01:48.353257+00:00,2097.662724,0.111187,0.135922,0.000078,buy,sell,1.0,CP-008,NaN,-0.210565,-0.228644,-0.036332,1,0.111187,0.156335,-233.396907,-328.079339
2,ETH-USDT,2026-02-08 11:02:47.061344+00:00,2101.229068,0.055957,0.069768,0.000039,buy,sell,1.0,CP-009,0.000369,-0.130799,-0.096947,-0.148163,1,0.055957,0.212292,-117.660696,-445.740034
3,ETH-USDT,2026-02-08 11:03:08.600562+00:00,2101.226492,0.013985,0.018190,0.000010,buy,sell,2.0,CP-004,0.000516,-0.039014,-0.028026,-0.026797,1,0.013985,0.226277,-29.406622,-475.146656
4,ETH-USDT,2026-02-08 11:03:09.695745+00:00,2100.766782,0.005594,0.007478,0.000004,buy,sell,2.0,CP-004,0.000489,-0.012532,-0.008112,-0.008654,1,0.005594,0.231871,-11.760273,-486.906929


In [168]:
ethusd_trade['s'] = ethusd_trade['taker_side'].map({'buy': 1, 'sell': -1})
ethusd_trade = ethusd_trade.dropna(subset=['trade_price','trade_amount'])

ethusd_trade["signed_amount"] = ethusd_trade["trade_amount"] * ethusd_trade["s"]
ethusd_trade["inventory"] = ethusd_trade["signed_amount"].cumsum()
ethusd_trade['cash_flow'] = ( - ethusd_trade['signed_amount'] * ethusd_trade['trade_price'] - ethusd_trade['fee_amount'] * ethusd_trade['trade_price'] )

ethusd_trade['cash'] = ethusd_trade['cash_flow'].cumsum()

final_price_eth = ethusd_price['mid_price'].iloc[-1]
final_inventory_eth = ethusd_trade['inventory'].iloc[-1]
final_cash_eth = ethusd_trade['cash'].iloc[-1]

accounting_pnl_eth = final_cash_eth + final_inventory_eth * final_price_eth

print(accounting_pnl_eth)

-1526.0161461257667


C:\Users\Startklar\AppData\Local\Temp\ipykernel_23780\1599896754.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ethusd_trade["signed_amount"] = ethusd_trade["trade_amount"] * ethusd_trade["s"]
C:\Users\Startklar\AppData\Local\Temp\ipykernel_23780\1599896754.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ethusd_trade["inventory"] = ethusd_trade["signed_amount"].cumsum()
C:\Users\Startklar\AppData\Local\Temp\ipykernel_23780\1599896754.py:6: SettingWithCopyWarning: 
A value is trying to be set on a c

The theoretical PnL tracks short-term performance after each trade, whereas the accounting pnl tracks actual cash flows over the span of the entire trading period. 

The accounting pnl accounts for what would happen if we help all the positions until the end. 